# CLI

Command-line interface backed by [python-fire](https://github.com/google/python-fire). Invoke via `python -m sugar <command> ...` or the `sugar` console script.

In [ ]:
#| default_exp cli

In [ ]:
#| export

import fire
from dataclasses import asdict
from sugar.chains import get_chain
from sugar.deposit import DepositQuote
from sugar.withdraw import Withdrawal
from sugar.helpers import ADDRESS_ZERO, normalize_address

In [ ]:
#| export

# Fire's literal_eval parses `0x...` as a hex int — unmangle before checksum/normalize.
def _addr(v): return v if v is None else normalize_address('0x' + format(v, '040x') if isinstance(v, int) else v)

def _new_pool_spec(c, token0, token1, pool_type, tick_spacing):
    t0, t1 = c.get_token(token0), c.get_token(token1)
    if t0 is None or t1 is None: raise SystemExit('token0 or token1 not known to Sugar')
    pool_type = str(pool_type).lower()
    if pool_type == 'cl':
        if tick_spacing is None: raise SystemExit('--pool-type cl requires --tick-spacing N')
        return c.pool_spec(t0, t1, tick_spacing=int(tick_spacing))
    if pool_type in ('stable', 'volatile'):
        if tick_spacing is not None: raise SystemExit('--tick-spacing is CL-only')
        return c.pool_spec(t0, t1, stable=(pool_type == 'stable'))
    raise SystemExit(f'invalid --pool-type: {pool_type} (expected cl/stable/volatile)')

def _resolve_pool(c, *, pool=None, token0=None, token1=None, pool_type=None, tick_spacing=None):
    """--pool ADDR (existing) XOR (--token0 --token1 --pool-type [--tick-spacing]) (new)."""
    pool, token0, token1 = _addr(pool), _addr(token0), _addr(token1)
    if pool is not None:
        if any(x is not None for x in (token0, token1, pool_type, tick_spacing)):
            raise SystemExit('--pool cannot be combined with --token0/--token1/--pool-type/--tick-spacing')
        p = c.get_pool_by_address(pool)
        if p is None: raise SystemExit(f'pool {pool} not found')
        return p
    if not token0 or not token1 or not pool_type:
        raise SystemExit('new pool requires --token0, --token1, and --pool-type {cl,stable,volatile}')
    return _new_pool_spec(c, token0, token1, pool_type, tick_spacing)

def _one_side(a0, a1, ctx):
    """Return the `amount_tokenN` kwarg for whichever side is set, raising if both/neither."""
    if (a0 is None) == (a1 is None): raise SystemExit(f'{ctx} requires exactly one of --amount0 / --amount1')
    return {'amount_token0': a0} if a0 is not None else {'amount_token1': a1}

def _build_quote(c, p, *, amount0, amount1, price_lower, price_upper, tick_lower, tick_upper, initial_price):
    """Resolve a DepositQuote for either basic or CL, new or existing pool."""
    is_new = p.lp == ADDRESS_ZERO
    a0 = p.token0.parse_units(float(amount0)) if amount0 is not None else None
    a1 = p.token1.parse_units(float(amount1)) if amount1 is not None else None
    if p.is_cl:
        kwargs = _one_side(a0, a1, 'CL deposit')
        if is_new and initial_price is None: raise SystemExit('new CL pool requires --initial-price')
        if initial_price is not None: kwargs['initial_price'] = float(initial_price)
        if tick_lower is not None or tick_upper is not None:
            kwargs['tick_lower'], kwargs['tick_upper'] = tick_lower, tick_upper
        else:
            kwargs['price_lower'] = float(price_lower) if price_lower is not None else None
            kwargs['price_upper'] = float(price_upper) if price_upper is not None else None
        return c.quote_concentrated_deposit(p, **kwargs)
    if any(x is not None for x in (price_lower, price_upper, tick_lower, tick_upper, initial_price)):
        raise SystemExit('basic deposits do not accept CL flags')
    if is_new:
        if a0 is None or a1 is None: raise SystemExit('new basic pool requires both --amount0 and --amount1')
        return DepositQuote(pool=p, amount_token0=a0, amount_token1=a1)
    return c.quote_basic_deposit(p, **_one_side(a0, a1, 'existing basic pool deposit'))
def _position_dict(p):
    """Compact dict for CLI output: asdict + overrides for the pool and amount fields."""
    t0, t1 = p.pool.token0, p.pool.token1
    return {**asdict(p),
            'pool': {'symbol': p.pool.symbol, 'lp': p.pool.lp, 'is_cl': p.pool.is_cl},
            'amount_token0': t0.to_float(p.amount_token0), 'amount_token1': t1.to_float(p.amount_token1),
            'staked_token0': t0.to_float(p.staked_token0), 'staked_token1': t1.to_float(p.staked_token1),
            'unstaked_earned0': t0.to_float(p.unstaked_earned0), 'unstaked_earned1': t1.to_float(p.unstaked_earned1)}

def _find_position(c, *, pool=None, position=None):
    """Find a wallet position. CL: --position=NFT_ID. Basic: --pool=LP_ADDR. --position=0 needs --pool."""
    pool = _addr(pool)
    if pool is None and position is None:
        raise SystemExit('requires --pool (basic) or --position (CL)')
    tid = int(position) if position is not None else 0
    if tid == 0 and pool is None:
        raise SystemExit('--position=0 is ambiguous (every basic position has id=0); pass --pool too')
    match = next((p for p in c.get_positions() if p.id == tid and (pool is None or p.pool.lp.lower() == pool.lower())), None)
    if match is None: raise SystemExit('position not found')
    return match
def _chain(chain):
    """Open a Chain context. `threading_max_workers=1` pins safe pagination for now."""
    return get_chain(str(chain), threading_max_workers=1)

def _result(r, broadcast):
    """Distill the SDK return: list-of-unsigned-tx-dicts in dry mode, focused receipt dict on broadcast."""
    if not broadcast: return r
    return {'tx': r.transactionHash.hex(), 'status': r.status, 'gas': r.gasUsed, 'block': r.blockNumber}

In [ ]:
#| export

class CLI:
    """Sugar SDK command-line interface.

    Default is dry-run (no broadcast). Pass --broadcast to send the tx.
    Amounts are human-readable (e.g. `0.5` → `0.5 * 10^decimals`).
    `--chain` takes a numeric chain id (10, 8453, 130, 1135).

    Examples:
        # preview a basic deposit
        python -m sugar deposit --chain=10 --pool=0xd25711... --amount0=0.0001 --amount1=1

        # list wallet positions
        python -m sugar positions --chain=10

        # withdraw 50% of a basic position
        python -m sugar withdraw --chain=10 --pool=0xd25711... --fraction=0.5 --broadcast"""

    def deposit(self, *, chain: int, pool: str = None,
                token0: str = None, token1: str = None, pool_type: str = None, tick_spacing=None,
                amount0=None, amount1=None,
                price_lower=None, price_upper=None, tick_lower=None, tick_upper=None,
                initial_price=None, slippage: float = 0.01, deadline_minutes: float = 30,
                broadcast: bool = False):
        """Deposit liquidity. Dry-run returns unsigned tx(s); --broadcast returns the receipt."""
        with _chain(chain) as c:
            p = _resolve_pool(c, pool=pool, token0=token0, token1=token1,
                              pool_type=pool_type, tick_spacing=tick_spacing)
            q = _build_quote(c, p, amount0=amount0, amount1=amount1,
                             price_lower=price_lower, price_upper=price_upper,
                             tick_lower=tick_lower, tick_upper=tick_upper,
                             initial_price=initial_price)
            return _result(c.deposit(q, delay_in_minutes=deadline_minutes, slippage=slippage, dry_run=not broadcast), broadcast)

    def positions(self, *, chain: int, owner: str = None):
        """List positions for `--owner` (defaults to the SUGAR_PK wallet)."""
        with _chain(chain) as c:
            return [_position_dict(p) for p in c.get_positions(_addr(owner))]

    def withdraw(self, *, chain: int, pool: str = None, position: int = None,
                 fraction: float = 1.0, burn: bool = False, collect: bool = True,
                 unwrap_native: bool = False, slippage: float = 0.01,
                 deadline_minutes: float = 30, broadcast: bool = False):
        """Withdraw a position. Identify by --pool (basic) or --position (CL). Default --fraction=1.0."""
        with _chain(chain) as c:
            p = _find_position(c, pool=pool, position=position)
            w = Withdrawal.from_position(p, fraction=float(fraction), burn=burn)
            return _result(c.withdraw(w, delay_in_minutes=deadline_minutes, slippage=slippage,
                                      collect=collect, unwrap_native=unwrap_native, dry_run=not broadcast), broadcast)

    def stake(self, *, chain: int, pool: str = None, position: int = None, broadcast: bool = False):
        """Stake an unstaked position into the pool's gauge."""
        with _chain(chain) as c:
            return _result(c.stake(_find_position(c, pool=pool, position=position), dry_run=not broadcast), broadcast)

    def unstake(self, *, chain: int, pool: str = None, position: int = None,
                amount: int = None, broadcast: bool = False):
        """Unstake from the pool's gauge. CL: full only. Basic: pass --amount for partial."""
        with _chain(chain) as c:
            return _result(c.unstake(_find_position(c, pool=pool, position=position), amount=amount, dry_run=not broadcast), broadcast)

    def claim_emissions(self, *, chain: int, pool: str = None, position: int = None, broadcast: bool = False):
        """Claim gauge emissions for a staked position."""
        with _chain(chain) as c:
            return _result(c.claim_emissions(_find_position(c, pool=pool, position=position), dry_run=not broadcast), broadcast)

    def claim_fees(self, *, chain: int, pool: str = None, position: int = None,
                   burn: bool = False, unwrap_native: bool = False, broadcast: bool = False):
        """Claim LP fees. CL: NFPM multicall (collect + optional unwrap/burn). Basic: pool.claimFees()."""
        with _chain(chain) as c:
            return _result(c.claim_fees(_find_position(c, pool=pool, position=position),
                                        burn=burn, unwrap_native=unwrap_native, dry_run=not broadcast), broadcast)

In [ ]:
#|hide
from fastcore.test import test_eq, test_fail
import inspect

# CLI surface: every subcommand exposes the expected kwargs and is keyword-only.
_subcommands = ['deposit', 'positions', 'withdraw', 'stake', 'unstake', 'claim_emissions', 'claim_fees']
for name in _subcommands:
    assert hasattr(CLI, name), f'CLI missing {name}'
    sig = inspect.signature(getattr(CLI, name))
    params = sig.parameters
    test_eq(params['chain'].default, inspect.Parameter.empty)  # chain is required
    if name != 'positions':
        test_eq(params['broadcast'].default, False)            # default is dry-run

# _addr: passes through strings, pads ints to 0x-prefixed 40-hex, returns None for None.
test_eq(_addr(None), None)
test_eq(_addr('0xd25711EdfBf747efCE181442Cc1D8F5F8fc8a0D3'), '0xd25711EdfBf747efCE181442Cc1D8F5F8fc8a0D3')
test_eq(_addr(int('0xd25711EdfBf747efCE181442Cc1D8F5F8fc8a0D3', 16)), '0xd25711EdfBf747efCE181442Cc1D8F5F8fc8a0D3')

# _one_side: dict-of-one OR exit on ambiguous.
test_eq(_one_side(100, None, 'ctx'), {'amount_token0': 100})
test_eq(_one_side(None, 200, 'ctx'), {'amount_token1': 200})
test_fail(lambda: _one_side(100, 200, 'ctx'), contains='exactly one')
test_fail(lambda: _one_side(None, None, 'ctx'), contains='exactly one')

# _result: distill receipt-like object on broadcast; pass through dry-run plan.
class _FakeReceipt:
    transactionHash = type('h', (), {'hex': lambda self: '0xabc'})()
    status, gasUsed, blockNumber = 1, 21000, 100

test_eq(_result(_FakeReceipt(), True), {'tx': '0xabc', 'status': 1, 'gas': 21000, 'block': 100})
test_eq(_result([{'to': '0xfoo'}], False), [{'to': '0xfoo'}])

In [ ]:
#| export

def main():
    from dotenv import load_dotenv
    load_dotenv()
    fire.Fire(CLI)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()